In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [2]:
MEAN_GRAY = 0.1307
STDDEV_GRAY = 0.3081

transforms = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize((MEAN_GRAY,), (STDDEV_GRAY,))])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transforms, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transforms)

In [3]:
BATCH_SIZE = 100
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [4]:
class CNN(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super(CNN, self).__init__(*args, **kwargs)

        self.cnn1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.batchnorm1 = nn.BatchNorm2d(8)

        self.relu = nn.ReLU()

        self.maxpool = nn.MaxPool2d(kernel_size=2)

        self.cnn2 = nn.Conv2d(in_channels=8, out_channels=32, kernel_size=3, stride=1, padding=2)
        self.batchnorm2 = nn.BatchNorm2d(32)

        self.fc1 = nn.Linear(2048, 600)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(600, 10)

    def forward(self, x):
        out = self.cnn1(x)
        out = self.batchnorm1(out)
        out = self.relu(out)
        out = self.maxpool(out)

        out = self.cnn2(out)
        out = self.batchnorm2(out)
        out = self.relu(out)
        out = self.maxpool(out)

        out = torch.flatten(out, 1)

        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        return out


In [5]:
model = CNN()
CUDA = torch.cuda.is_available()
if CUDA:
    model = model.cuda()

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [6]:
# UNDERSTAND. 

'''iteration = 0
correct = 0

for i, (inputs, labels) in enumerate(train_loader):
    if CUDA:
        inputs = inputs.cuda()
        labels = labels.cuda()

    print("input shape = ", inputs.shape)
    print("labels shape = ", labels.shape)

    output = model(inputs)
    print("output shape = ", output.shape)
    _, predicted = torch.max(output, 1)

    print("predicted shape = ", predicted.shape )
    correct += (predicted == labels).sum()
    print(int(correct))
    break'''

'iteration = 0\ncorrect = 0\n\nfor i, (inputs, labels) in enumerate(train_loader):\n    if CUDA:\n        inputs = inputs.cuda()\n        labels = labels.cuda()\n\n    print("input shape = ", inputs.shape)\n    print("labels shape = ", labels.shape)\n\n    output = model(inputs)\n    print("output shape = ", output.shape)\n    _, predicted = torch.max(output, 1)\n\n    print("predicted shape = ", predicted.shape )\n    correct += (predicted == labels).sum()\n    print(int(correct))\n    break'

In [7]:
# training.

EPOCHS = 10
train_loss = []
train_accuracy = []
test_loss_list = []
test_accuracy = []

for epoch in range(EPOCHS):
    correct = 0
    iteraions = 0
    iter_loss = 0

    model.train()

    for i, (inputs, labels) in enumerate(train_loader):
        
        if CUDA:
            inputs = inputs.cuda()
            labels = labels.cuda()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        iter_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        iteraions += 1

    train_loss.append(iter_loss/iteraions)
    train_accuracy.append(100*correct/len(train_dataset))

    # testing phase
    test_loss = 0.0 
    correct = 0
    iterations = 0

    model.eval()

    for i, (inputs, labels) in enumerate(test_loader):
        if CUDA: 
            inputs = inputs.cuda()
            labels = labels.cuda()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        iterations += 1

    test_loss_list.append(test_loss/iterations)
    test_accuracy.append(100*correct/len(test_dataset))

    print(f"Epoch{epoch}/{EPOCHS}, Training loss: {train_loss[-1]:.3f}, Training Accuracy: {train_accuracy[-1]:.3f}, Testing Loss: {test_loss_list[-1]:.3f}, Testing accuracy: {test_accuracy[-1]:.3f}")
    

Epoch0/10, Training loss: 0.517, Training Accuracy: 89.222, Testing Loss: 0.071, Testing accuracy: 97.830
Epoch1/10, Training loss: 0.139, Training Accuracy: 95.898, Testing Loss: 0.058, Testing accuracy: 98.320
Epoch2/10, Training loss: 0.104, Training Accuracy: 96.962, Testing Loss: 0.051, Testing accuracy: 98.380
Epoch3/10, Training loss: 0.091, Training Accuracy: 97.348, Testing Loss: 0.034, Testing accuracy: 98.910
Epoch4/10, Training loss: 0.075, Training Accuracy: 97.818, Testing Loss: 0.040, Testing accuracy: 98.820
Epoch5/10, Training loss: 0.074, Training Accuracy: 97.822, Testing Loss: 0.041, Testing accuracy: 98.820
Epoch6/10, Training loss: 0.072, Training Accuracy: 97.850, Testing Loss: 0.041, Testing accuracy: 98.780
Epoch7/10, Training loss: 0.070, Training Accuracy: 97.925, Testing Loss: 0.040, Testing accuracy: 98.760
Epoch8/10, Training loss: 0.067, Training Accuracy: 98.015, Testing Loss: 0.038, Testing accuracy: 98.780
Epoch9/10, Training loss: 0.064, Training Accu

In [8]:
print(train_accuracy)
print(test_accuracy)

[89.22166666666666, 95.89833333333333, 96.96166666666667, 97.34833333333333, 97.81833333333333, 97.82166666666667, 97.85, 97.925, 98.015, 98.15]
[97.83, 98.32, 98.38, 98.91, 98.82, 98.82, 98.78, 98.76, 98.78, 99.0]
